# Demand Signal Feature Store — Capstone Project

**Business problem:** Demand forecasts run on structured ERP data (shipments, orders, inventory) and miss early warnings buried in supplier notes. By the time a delay shows up in the numbers, the forecast has already been wrong for a cycle.

**What we build:** A pipeline that extracts delay signals from supplier notes, merges them with ERP features, and proves the enhanced forecast has lower error (MAPE) than the ERP-only baseline.

**Important:** Observed sales ≈ demand proxy when inventory is sufficient. During stockouts, sales underestimates true demand. We use future sales as the forecast target and flag stockout observations.

## 0. Setup

Install dependencies and create required directories.

In [ ]:
import os
os.makedirs("../data", exist_ok=True)

!pip install -q -r ../requirements.txt

print("Dependencies installed and data/ directory ready.")

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor

from src.data_generator import generate_all_data, generate_ground_truth_extractions, HOLIDAY_SET
from src.llm_extractor import get_client, extract_batch
from src.drift import calculate_psi_for_features, calculate_csi

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)

print("Setup complete.")

## 1. Generate Synthetic Data

We simulate 540 days (~18 months) of supply chain data for 5 SKUs and 4 suppliers:
- **Daily sales & inventory** — one row per SKU per day
- **Purchase orders** — with promised, actual, and ERP-updated delivery dates
- **Supplier notes** — delay warnings + routine check-ins

The data is internally consistent: when a supplier is delayed, inventory drops, and if it runs out, sales fall below true demand (stockout).

In [ ]:
daily_df, po_df, notes_df = generate_all_data(n_days=540, start_date="2024-01-01", seed=42)

print(f"Daily records: {len(daily_df)} ({daily_df['sku_id'].nunique()} SKUs x {daily_df['date'].nunique()} days)")
print(f"Purchase orders: {len(po_df)} ({po_df['is_delayed'].sum()} delayed, {(~po_df['is_delayed']).sum()} on time)")
print(f"Supplier notes: {len(notes_df)} ({(notes_df['note_type']=='delay_warning').sum()} delay warnings, {(notes_df['note_type']=='routine').sum()} routine)")
print(f"Date range: {daily_df['date'].min().date()} to {daily_df['date'].max().date()}")
print(f"Stockout rate: {daily_df['stockout_flag'].mean():.1%}")

In [ ]:
# Sample ERP daily data
print("=== Daily Sales & Inventory (sample) ===")
daily_df.head(10)

In [ ]:
# Sample purchase orders
print("=== Purchase Orders (sample) ===")
po_df.head(10)

In [ ]:
# Sample supplier notes
print("=== Supplier Notes (samples) ===")
for _, row in notes_df.head(6).iterrows():
    tag = "DELAY" if row["note_type"] == "delay_warning" else "ROUTINE"
    print(f"\n[{row['note_date'].strftime('%Y-%m-%d')}] [{tag}] {row['supplier_id']} / {row['po_id']}:")
    print(f"  {row['note_text']}")

In [ ]:
# Save raw data
daily_df.to_csv("../data/daily_sales_inventory.csv", index=False)
po_df.to_csv("../data/purchase_orders.csv", index=False)
notes_df.to_csv("../data/supplier_notes.csv", index=False)
print("Raw data saved to ../data/")

## 2. Extract Delay Signals from Supplier Notes

We use an LLM (via OpenRouter) to read each supplier note and extract:
- `early_delay_flag`: 1 if the note warns of a delay, 0 otherwise
- `expected_delay_days`: how many days of delay the note mentions
- `delay_reason`: why the delay is happening

Each extraction carries lineage: timestamp, model version, source.

Set `USE_LLM = False` to use ground-truth fallback (no API needed).

In [ ]:
USE_LLM = True  # Set to False for ground-truth fallback (no API needed)

if USE_LLM:
    print("Using OpenRouter LLM for extraction...")
    client = get_client()
    extracted_df = extract_batch(client, notes_df, batch_delay=0.3)

    # Patch failed extractions with ground truth
    gt_df = generate_ground_truth_extractions(notes_df, po_df)
    if "extraction_error" in extracted_df.columns:
        failed = extracted_df["extraction_error"].notna()
        n_failed = failed.sum()
        if n_failed > 0:
            print(f"\n  Patching {n_failed} failed extractions with ground truth...")
            for col in ["early_delay_flag", "expected_delay_days", "delay_reason"]:
                extracted_df.loc[failed, col] = gt_df.loc[failed, col].values
            extracted_df.loc[failed, "source"] = "ground_truth_fallback"
else:
    print("Using ground-truth extraction (no API call)...")
    extracted_df = generate_ground_truth_extractions(notes_df, po_df)

print(f"\nExtracted {len(extracted_df)} notes")
print(f"Delay warnings detected: {extracted_df['early_delay_flag'].sum()}")
print(f"Source: {extracted_df['source'].iloc[0]}")
extracted_df.head(10)

In [ ]:
# Spot-check: compare note text with extraction
print("=== Extraction Spot-Check ===")
for _, ext in extracted_df.head(5).iterrows():
    note = notes_df[notes_df["note_id"] == ext["note_id"]].iloc[0]
    print(f"\nNote: {note['note_text'][:120]}...")
    print(f"  early_delay_flag: {ext['early_delay_flag']}")
    print(f"  expected_delay_days: {ext['expected_delay_days']}")
    print(f"  delay_reason: {ext['delay_reason']}")

In [ ]:
extracted_df.to_csv("../data/extracted_signals.csv", index=False)
print("Extracted signals saved.")

## 3. Build ERP Features

For each (date, SKU) row, we compute features that would be available BEFORE that date:
- **Sales lags**: Sales_Lag_1, Sales_Lag_7, Sales_Lag_28
- **Rolling averages**: Rolling_Avg_7, Rolling_Avg_28
- **Inventory**: On_Hand_Inventory, Available_Inventory, Stockout_Flag, Inventory_Cover
- **Supply**: Open_PO_Qty, Days_Until_Expected_Delivery
- **Historical supplier delay**: average delay from past completed POs
- **Calendar**: Day_of_Week, Month, Holiday

In [ ]:
df = daily_df.copy().sort_values(["sku_id", "date"]).reset_index(drop=True)

# --- Sales lag features (shifted to avoid leakage) ---
for lag in [1, 7, 28]:
    df[f"sales_lag_{lag}"] = df.groupby("sku_id")["sales"].shift(lag)

# --- Rolling averages (shifted by 1 so we don't include today) ---
df["rolling_avg_7"] = df.groupby("sku_id")["sales"].transform(
    lambda x: x.shift(1).rolling(7, min_periods=3).mean()
)
df["rolling_avg_28"] = df.groupby("sku_id")["sales"].transform(
    lambda x: x.shift(1).rolling(28, min_periods=7).mean()
)

# --- Inventory features ---
df["available_inventory"] = df["on_hand_inventory"] + df["open_po_qty"]
df["inventory_cover"] = df["on_hand_inventory"] / df["rolling_avg_7"].clip(lower=1)

# --- Calendar features ---
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["holiday"] = df["date"].isin(HOLIDAY_SET).astype(int)

# --- Historical average supplier delay ---
# For each date, compute running avg delay from POs that arrived before that date
po_completed = po_df.sort_values("actual_date")
supplier_delay_history = []
for sid in po_df["supplier_id"].unique():
    sp = po_completed[po_completed["supplier_id"] == sid]
    running = []
    for _, po in sp.iterrows():
        running.append(po["delay_days"])
        supplier_delay_history.append({
            "supplier_id": sid,
            "as_of_date": po["actual_date"],
            "historical_avg_supplier_delay": np.mean(running),
        })

delay_hist = pd.DataFrame(supplier_delay_history).sort_values("as_of_date")

# Merge using as-of join per supplier
df["historical_avg_supplier_delay"] = 0.0
for sid in df["supplier_id"].unique():
    mask = df["supplier_id"] == sid
    sub = df.loc[mask, ["date"]].copy()
    hist = delay_hist[delay_hist["supplier_id"] == sid].rename(columns={"as_of_date": "date"})
    if hist.empty:
        continue
    merged = pd.merge_asof(sub.sort_values("date"), hist[["date", "historical_avg_supplier_delay"]], on="date", direction="backward")
    df.loc[mask, "historical_avg_supplier_delay"] = merged["historical_avg_supplier_delay"].fillna(0).values

# --- Target: next-day sales (demand proxy) ---
df["target_next_day_sales"] = df.groupby("sku_id")["sales"].shift(-1)

print(f"Feature matrix: {df.shape}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

## 4. Merge Supplier Note Signals

For each (date, SKU), we check: is there an active delay warning from a supplier note?

**Temporal constraint:** A warning is only active AFTER the note was written and BEFORE the PO arrives. This prevents data leakage.

This is the key feature: `Early_Delay_Flag` provides information that the ERP doesn't have yet.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Build active-warning windows from extracted signals
delay_extractions = extracted_df[extracted_df["early_delay_flag"] == 1]

active_warnings = []
for _, ext in delay_extractions.iterrows():
    note = notes_df[notes_df["note_id"] == ext["note_id"]]
    if note.empty:
        continue
    note = note.iloc[0]
    po = po_df[po_df["po_id"] == ext["po_id"]]
    if po.empty:
        continue
    po = po.iloc[0]

    note_date = note["note_date"]
    po_arrival = po["actual_date"]

    for d in pd.date_range(note_date, po_arrival - pd.Timedelta(days=1)):
        active_warnings.append({
            "date": d,
            "sku_id": po["sku_id"],
            "early_delay_flag": 1,
            "expected_delay_days": int(ext["expected_delay_days"]),
            "delay_reason": str(ext["delay_reason"]),
        })

if active_warnings:
    warnings_df = pd.DataFrame(active_warnings).drop_duplicates(subset=["date", "sku_id"])
    df = df.merge(warnings_df, on=["date", "sku_id"], how="left")
    df["early_delay_flag"] = df["early_delay_flag"].fillna(0).astype(int)
    df["expected_delay_days"] = df["expected_delay_days"].fillna(0).astype(int)
    df["delay_reason"] = df["delay_reason"].fillna("none")
else:
    df["early_delay_flag"] = 0
    df["expected_delay_days"] = 0
    df["delay_reason"] = "none"

# Label-encode delay_reason so XGBoost can use it
le = LabelEncoder()
df["delay_reason_encoded"] = le.fit_transform(df["delay_reason"])

print(f"Days with active delay warning: {df['early_delay_flag'].sum()} ({df['early_delay_flag'].mean():.1%})")
print(f"Delay reasons: {list(le.classes_)}")
print(f"\nSample rows with delay warnings:")
df[df["early_delay_flag"] == 1].head(5)[["date", "sku_id", "sales", "on_hand_inventory",
    "early_delay_flag", "expected_delay_days", "delay_reason"]]

## 5. Feature Store with Lineage

A simple Parquet-based feature store with metadata: feature name, source, timestamps.

In [ ]:
from datetime import datetime, timezone

# Define feature lineage
feature_lineage = pd.DataFrame([
    {"feature_name": "sales_lag_1", "source": "erp", "description": "Sales 1 day ago"},
    {"feature_name": "sales_lag_7", "source": "erp", "description": "Sales 7 days ago"},
    {"feature_name": "sales_lag_28", "source": "erp", "description": "Sales 28 days ago"},
    {"feature_name": "rolling_avg_7", "source": "erp", "description": "7-day rolling avg sales"},
    {"feature_name": "rolling_avg_28", "source": "erp", "description": "28-day rolling avg sales"},
    {"feature_name": "on_hand_inventory", "source": "erp", "description": "Current inventory level"},
    {"feature_name": "available_inventory", "source": "erp", "description": "On-hand inventory + open PO qty"},
    {"feature_name": "stockout_flag", "source": "erp", "description": "1 if inventory was zero"},
    {"feature_name": "inventory_cover", "source": "erp", "description": "Days of inventory remaining"},
    {"feature_name": "open_po_qty", "source": "erp", "description": "Units in open purchase orders"},
    {"feature_name": "days_until_expected_delivery", "source": "erp", "description": "Days until next PO arrives"},
    {"feature_name": "historical_avg_supplier_delay", "source": "erp", "description": "Running avg delay from past POs"},
    {"feature_name": "day_of_week", "source": "calendar", "description": "0=Mon, 6=Sun"},
    {"feature_name": "month", "source": "calendar", "description": "Month (1-12)"},
    {"feature_name": "holiday", "source": "calendar", "description": "1 if US holiday"},
    {"feature_name": "early_delay_flag", "source": "supplier_note", "description": "1 if note warns of delay (BEFORE ERP knows)"},
    {"feature_name": "expected_delay_days", "source": "supplier_note", "description": "Delay days from note (BEFORE ERP knows)"},
    {"feature_name": "delay_reason_encoded", "source": "supplier_note", "description": "Encoded delay reason from note"},
])

now = datetime.now(timezone.utc).isoformat()
feature_lineage["source_timestamp"] = daily_df["date"].max().isoformat()
feature_lineage["created_timestamp"] = now
feature_lineage["freshness_status"] = "fresh"

print("=== Feature Lineage ===")
feature_lineage

In [ ]:
# Save feature store as Parquet
df.to_parquet("../data/feature_store.parquet", index=False)
feature_lineage.to_csv("../data/feature_lineage.csv", index=False)
print(f"Feature store saved: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Feature lineage saved: {len(feature_lineage)} features tracked")

## 6. Prepare Training Data

**Leakage prevention:**
- All features use only information available BEFORE the target date
- Lag features are shifted (no same-day data)
- Note signals only active after note_date
- Chronological train/test split (no random shuffle)

In [ ]:
# Define feature sets
ERP_FEATURES = [
    "sales_lag_1", "sales_lag_7", "sales_lag_28",
    "rolling_avg_7", "rolling_avg_28",
    "on_hand_inventory", "available_inventory", "stockout_flag", "inventory_cover",
    "open_po_qty", "days_until_expected_delivery",
    "historical_avg_supplier_delay",
    "day_of_week", "month", "holiday",
]

NOTE_FEATURES = [
    "early_delay_flag",
    "expected_delay_days",
]

ENHANCED_FEATURES = ERP_FEATURES + NOTE_FEATURES
TARGET = "target_next_day_sales"

print(f"Baseline features ({len(ERP_FEATURES)}): {ERP_FEATURES}")
print(f"\nEnhanced features ({len(ENHANCED_FEATURES)}): adds {NOTE_FEATURES}")
print(f"\nNote: delay_reason is extracted and stored in the feature store for lineage,")
print(f"but excluded from the model as categorical text is noisy with live LLM extraction.")

In [ ]:
# Drop rows with NaN (from lag warmup period + last row with no target)
model_df = df.dropna(subset=[TARGET] + ERP_FEATURES).copy()
model_df = model_df.sort_values(["date", "sku_id"]).reset_index(drop=True)

# Fill any remaining NaN in note features
model_df["days_until_expected_delivery"] = model_df["days_until_expected_delivery"].fillna(-1)

# Chronological split: last 20% as test
all_dates = sorted(model_df["date"].unique())
split_idx = int(len(all_dates) * 0.8)
train_dates = all_dates[:split_idx]
test_dates = all_dates[split_idx:]

train_df = model_df[model_df["date"].isin(train_dates)]
test_df = model_df[model_df["date"].isin(test_dates)]

print(f"Total usable rows: {len(model_df)}")
print(f"Train: {len(train_df)} rows ({train_dates[0].strftime('%Y-%m-%d')} to {train_dates[-1].strftime('%Y-%m-%d')})")
print(f"Test:  {len(test_df)} rows ({test_dates[0].strftime('%Y-%m-%d')} to {test_dates[-1].strftime('%Y-%m-%d')})")
print(f"\nDelay warnings in test set: {test_df['early_delay_flag'].sum()} ({test_df['early_delay_flag'].mean():.1%})")

In [ ]:
# Final training data sample
print("=== Training Data Sample ===")
display_cols = ["date", "sku_id", "sales_lag_1", "rolling_avg_7", "on_hand_inventory",
                "stockout_flag", "early_delay_flag",
                "expected_delay_days", "delay_reason", "target_next_day_sales"]
model_df[display_cols].head(10)

## 7. Train Baseline Model (ERP Only)

**Model: XGBoost** — fast, handles tabular data well, provides feature importance, easy to explain.

The baseline uses only structured ERP features. No supplier note information.

In [ ]:
X_train_base = train_df[ERP_FEATURES]
X_test_base = test_df[ERP_FEATURES]
y_train = train_df[TARGET]
y_test = test_df[TARGET]

baseline_model = XGBRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
)
baseline_model.fit(X_train_base, y_train)
y_pred_base = baseline_model.predict(X_test_base)

# Clip predictions to non-negative
y_pred_base = np.clip(y_pred_base, 0, None)

# MAPE needs non-zero actuals — filter or use safe version
mask_nonzero = y_test > 0
mape_base = mean_absolute_percentage_error(y_test[mask_nonzero], y_pred_base[mask_nonzero])
mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = root_mean_squared_error(y_test, y_pred_base)

print("=== Baseline Model (ERP Only) ===")
print(f"  MAPE: {mape_base:.2%}")
print(f"  MAE:  {mae_base:.2f}")
print(f"  RMSE: {rmse_base:.2f}")

## 8. Train Enhanced Model (ERP + Supplier Note Signals)

Same model architecture, but now includes `Early_Delay_Flag`, `Expected_Delay_Days`, and `Delay_Reason` — the early warning signals extracted from supplier notes.

In [ ]:
X_train_enh = train_df[ENHANCED_FEATURES]
X_test_enh = test_df[ENHANCED_FEATURES]

enhanced_model = XGBRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
)
enhanced_model.fit(X_train_enh, y_train)
y_pred_enh = enhanced_model.predict(X_test_enh)
y_pred_enh = np.clip(y_pred_enh, 0, None)

mape_enh = mean_absolute_percentage_error(y_test[mask_nonzero], y_pred_enh[mask_nonzero])
mae_enh = mean_absolute_error(y_test, y_pred_enh)
rmse_enh = root_mean_squared_error(y_test, y_pred_enh)

print("=== Enhanced Model (ERP + Supplier Notes) ===")
print(f"  MAPE: {mape_enh:.2%}")
print(f"  MAE:  {mae_enh:.2f}")
print(f"  RMSE: {rmse_enh:.2f}")

## 9. Compare Models

In [ ]:
mape_change = (mape_enh - mape_base) / mape_base * 100

comparison = pd.DataFrame({
    "Model": ["Baseline (ERP only)", "Enhanced (ERP + supplier notes)"],
    "Features": [f"{len(ERP_FEATURES)} ERP", f"{len(ERP_FEATURES)} ERP + {len(NOTE_FEATURES)} note signals"],
    "MAPE": [f"{mape_base:.2%}", f"{mape_enh:.2%}"],
    "MAE": [f"{mae_base:.2f}", f"{mae_enh:.2f}"],
    "RMSE": [f"{rmse_base:.2f}", f"{rmse_enh:.2f}"],
})

print("="*70)
print("FORECAST PERFORMANCE COMPARISON")
print("="*70)
print(comparison.to_string(index=False))
print(f"\nMAPE change: {mape_change:+.1f}% {'(improved)' if mape_change < 0 else '(no improvement)'}")
print("="*70)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAPE bar chart
models = ["Baseline\n(ERP only)", "Enhanced\n(ERP + Notes)"]
mapes = [mape_base, mape_enh]
colors = ["#d4726a", "#5b9f5b"]
axes[0].bar(models, mapes, color=colors, width=0.5)
axes[0].set_ylabel("MAPE")
axes[0].set_title("MAPE Comparison")
for i, v in enumerate(mapes):
    axes[0].text(i, v + 0.005, f"{v:.2%}", ha="center", fontweight="bold")

# Actual vs predicted — Baseline
axes[1].scatter(y_test, y_pred_base, alpha=0.3, s=10, color=colors[0])
axes[1].plot([0, y_test.max()], [0, y_test.max()], "k--", alpha=0.5)
axes[1].set_xlabel("Actual Sales")
axes[1].set_ylabel("Predicted Sales")
axes[1].set_title("Baseline: Actual vs Predicted")

# Actual vs predicted — Enhanced
axes[2].scatter(y_test, y_pred_enh, alpha=0.3, s=10, color=colors[1])
axes[2].plot([0, y_test.max()], [0, y_test.max()], "k--", alpha=0.5)
axes[2].set_xlabel("Actual Sales")
axes[2].set_ylabel("Predicted Sales")
axes[2].set_title("Enhanced: Actual vs Predicted")

plt.tight_layout()
plt.savefig("../data/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "feature": ENHANCED_FEATURES,
    "importance": enhanced_model.feature_importances_,
}).sort_values("importance", ascending=True)

importance["is_note_feature"] = importance["feature"].isin(NOTE_FEATURES)

fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = ["#5b9f5b" if nf else "#6a8fc4" for nf in importance["is_note_feature"]]
ax.barh(importance["feature"], importance["importance"], color=bar_colors)
ax.set_xlabel("Feature Importance")
ax.set_title("Feature Importance — Enhanced Model\n(Green = supplier note signals, Blue = ERP features)")
plt.tight_layout()
plt.savefig("../data/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 features:")
print(importance.tail(5)[["feature", "importance"]].to_string(index=False))

## 11. Feature Drift Detection (PSI)

PSI (Population Stability Index) measures how much a feature's distribution has shifted between the training (reference) period and the test (current) period.

- PSI < 0.1: stable
- 0.1 ≤ PSI < 0.2: moderate drift
- PSI ≥ 0.2: significant drift

We focus on the supplier-note features since those are the ones that shift when supplier language changes.

In [ ]:
drift_features = [
    "sales_lag_1", "rolling_avg_7", "on_hand_inventory", "inventory_cover",
    "historical_avg_supplier_delay",
    "early_delay_flag", "expected_delay_days",
]

psi_results = calculate_psi_for_features(train_df, test_df, drift_features)
print("=== PSI Drift Detection ===")
print(psi_results.to_string(index=False))

In [ ]:
psi_plot = psi_results.dropna(subset=["psi"]).sort_values("psi", ascending=True)

psi_colors = []
for _, row in psi_plot.iterrows():
    if row["status"] == "stable":
        psi_colors.append("#5b9f5b")
    elif row["status"] == "moderate_drift":
        psi_colors.append("#d4a44a")
    else:
        psi_colors.append("#d4726a")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(psi_plot["feature"], psi_plot["psi"], color=psi_colors)
ax.axvline(x=0.1, color="orange", linestyle="--", alpha=0.7, label="Moderate (0.1)")
ax.axvline(x=0.2, color="red", linestyle="--", alpha=0.7, label="Significant (0.2)")
ax.set_xlabel("PSI")
ax.set_title("Feature Drift: Training vs Test Period")
ax.legend()
plt.tight_layout()
plt.savefig("../data/psi_drift.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Freshness SLA

Simple check: how recently were the features updated? For a production system, features older than 24 hours would be flagged stale.

In [ ]:
from config.settings import FRESHNESS_SLA_HOURS

now = datetime.now(timezone.utc)
lineage = feature_lineage.copy()
lineage["hours_since_creation"] = 0.1  # just created
lineage["sla_met"] = lineage["hours_since_creation"] <= FRESHNESS_SLA_HOURS

print(f"=== Freshness SLA ({FRESHNESS_SLA_HOURS}h) ===")
print(f"Features meeting SLA: {lineage['sla_met'].sum()}/{len(lineage)}")
print(f"\nNote: In production, ERP features refresh daily. Supplier-note features")
print(f"refresh when new notes arrive and are extracted. Stale features would be flagged.")
lineage[["feature_name", "source", "freshness_status", "sla_met"]]

## 13. Summary & Conclusion

### What we built
A feature pipeline that extracts early supplier delay signals from unstructured notes and merges them with structured ERP data to improve demand forecasting.

### Key results

In [ ]:
print("="*60)
print("PROJECT SUMMARY")
print("="*60)
print(f"\nData: {daily_df['date'].nunique()} days, {daily_df['sku_id'].nunique()} SKUs, {len(po_df)} POs, {len(notes_df)} notes")
print(f"Extraction: {len(extracted_df)} notes processed, {extracted_df['early_delay_flag'].sum()} delay warnings found")
print(f"Source: {extracted_df['source'].iloc[0]}")
print(f"\n--- Forecast Performance ---")
print(f"Baseline MAPE (ERP only):           {mape_base:.2%}")
print(f"Enhanced MAPE (ERP + note signals):  {mape_enh:.2%}")
print(f"Change:                              {mape_change:+.1f}%")
print(f"\n--- Feature Store ---")
print(f"Total features: {len(ENHANCED_FEATURES)} ({len(ERP_FEATURES)} ERP + {len(NOTE_FEATURES)} note-derived)")
print(f"Storage: Parquet with lineage metadata")
print(f"Freshness SLA: {FRESHNESS_SLA_HOURS}h")
print(f"\n--- Drift Detection ---")
n_drifting = len(psi_results[psi_results['status'] != 'stable'])
print(f"Features drifting: {n_drifting}/{len(drift_features)}")
print(f"\n--- Interpretation ---")
print(f"The supplier note provides early warning of delays BEFORE the ERP is updated.")
print(f"This does NOT mean delays reduce demand — it means delays may constrain")
print(f"future supply/availability, affecting realized sales. The model uses this")
print(f"signal to predict when observed sales will drop due to supply constraints.")
print("="*60)